In [ ]:
using PyPlot
using LinearAlgebra
using SparseArrays

The linear wave equation:
$$
\begin{cases}
  \partial_t\,u \,+\,\partial_x\,v\,&=\,0,
  \\[0.3em]
  \partial_t\,v \,+c^2\,\partial_x\,u\,&=\,0.
\end{cases}
$$

In [ ]:
# the domain Ω
Ω = [0.0, 4.0]
a, b = Ω

# spatial discretization
k = 8
h = (b - a) / (2^k + 1)
xs = [a + h * (i - 1) for i in 1:(2^k+2)]

# Final time
T = 16.
output_granularity = 0.1
CFL = 0.1
out = 1.0e-5

;

In [ ]:
c = 0.25

function dirichlet(t, x)
    u = 0.25 * exp(-4 * (x-2)^2)
    v = 0
    return u, v
end

function f(t, x)
    return 0, 0
end

;

In [ ]:
c = 0.25

function dirichlet(t, x)
    u = 1 + exp(-8 * (x-2)^2)
    v = 0
    return u, v
end

function f(t, x)
    return 0, 0
end

;

In [ ]:
c = 0.25

function dirichlet(t, x)
    u = sin(π * (x - c * t))
    v = c * sin(π * (x - c * t))
    return u, v
end

function f(t, x)
    return 0, 0
end

;

In [ ]:
function euler_step_linear(Ω, k, CFL, U_old, t_old)
    a, b = Ω
    h = (b - a) / (2^k + 1)

    u_old = U_old[1,:]
    v_old = U_old[2,:]

    τ = CFL * h
    stabilization = 1.

    u_new = zeros(2^k + 2)
    v_new = zeros(2^k + 2)

    #
    # Central difference for interior degrees of freedom:
    #
    
    for i in 2 : 2^k + 1
      u_new[i] = u_old[i] - τ / (2.0 * h) * (v_old[i+1] - v_old[i-1])
      u_prime = 1.0 / (2.0 * h) * (u_old[i+1] - u_old[i-1])
      v_new[i] = v_old[i] - c^2 * τ * u_prime

      u_new[i] = u_new[i] + stabilization * τ * (u_old[i+1] - 2.0 * u_old[i] + u_old[i-1])
      v_new[i] = v_new[i] + stabilization * τ * (v_old[i+1] - 2.0 * v_old[i] + v_old[i-1])
    end

    #
    # One-sided difference for boundary degrees of freedom:
    #
    
    u_new[1] = u_old[1] - τ / (1.0 * h) * (v_old[2] - v_old[1])
    u_prime = 1.0 / (1.0 * h) * (u_old[2] - u_old[1])
    v_new[1] = v_old[1] - c^2 * τ * u_prime

    N = 2^k + 2
    u_new[N] = u_old[N] - τ / (1.0 * h) * (v_old[N] - v_old[N-1])
    u_prime = 1.0 / (1.0 * h) * (u_old[N] - u_old[N-1])
    v_new[N] = v_old[N] - c^2 * τ * u_prime

    #
    # Boundary value post-processing:
    #

    # reflecting boundary conditions:
    #v_new[1] = 0.
    #v_new[N] = 0.

    # Dirichlet boundary conditions:
    #u_new[1], v_new[1] = dirichlet(t + τ, xs[1])
    #u_new[N], v_new[N] = dirichlet(t + τ, xs[N])

    # Periodic boundary conditions:
    #f_1 = f(t, xs[1])
    #u_new[1] = u_old[1] - τ / (2.0 * h) * (v_old[2] - v_old[N-1])
    #u_prime = 1.0 / (2.0 * h) * (u_old[2] - u_old[N-1])
    #v_new[1] = v_old[1] - c^2 * τ * u_prime
    #u_new[1] = u_new[1] + stabilization * τ * (u_old[2] - 2.0 * u_old[1] + u_old[N-1])
    #v_new[1] = v_new[1] + stabilization * τ * (v_old[2] - 2.0 * v_old[1] + v_old[N-1])
    #u_new[N] = u_new[1]
    #v_new[N] = v_old[1]
    
    return vcat(u_new', v_new'), τ
end

In [ ]:
U = zeros(2, 2^k + 2) # The two-component state vector [u, v]
U[1,:] = [dirichlet(0, x)[1] for x in xs]
U[2,:] = [dirichlet(0, x)[2] for x in xs]

#yrange = [1, 2] # for plotting

print("output at initial time t = 0\n")
plt.figure(figsize=(10,7))
plot(xs, U[1,:])
plt.xlim(Ω)
#plt.ylim(yrange)
plt.savefig("output-" * lpad(0,4,"0") * ".png")
plt.close()
output_cycle = 1

t = 0
while t < T
    U, τ = euler_step_linear(Ω, k, CFL, U, t)
    if(τ < out * T)
        print("We didn't make it.\n")
        break
    end
        
    t = t + τ

    if (output_cycle * output_granularity < t)
        print("output at time t = ", t, " (τ = ", τ, ")\n")
        plt.figure(figsize=(10,7))
        plot(xs, U[1,:])
        plt.xlim(Ω)
        #plt.ylim(yrange)
        plt.savefig("output-" * lpad(output_cycle,4,"0") * ".png")
        plt.close()
        output_cycle = output_cycle + 1
    end
end;